# ⚡ NeoScan AI — Notebook 3: Late Fusion Phase 2 (Multimodal)
**Project:** NeoScan AI — Neonatal LUS RDS Triage | SRHU
**Notebook:** 3 of 3 — Multimodal Late Fusion + Gradio Demo

---
### ✅ FIRST TIME: Runtime → T4 GPU → Run All
### 🔁 RETURNING AFTER TRAINING: Run Cells 1, 2, 4 (define models), 8 (HF login + load), then Cell 10 (Gradio demo)

---
Architecture:
- ImageEncoder: MobileNetV3-Small features → avgpool → 576-d vector
- ClinicalEncoder: 8 → 64 → 32-d vector
- FusionHead: concat(576+32=608) → Linear(128) → ReLU → Dropout(0.3) → Linear(3)

Output classes: Normal (0) | Moderate RDS (1) | High RDS (2)
Expected time on T4: ~20-30 min

In [ ]:
!pip install -q gradio scikit-learn pandas numpy torch torchvision pillow tqdm huggingface_hub

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Load pretrained artifacts from HuggingFace
# USER ACTION: Replace YOUR_HF_USERNAME with your actual HuggingFace username below

from huggingface_hub import hf_hub_download, login
from google.colab import userdata
import torch

token = userdata.get('HF_TOKEN')
login(token=token)

# Download Phase 1 image encoder weights
phase1_path = hf_hub_download(
    repo_id="YOUR_HF_USERNAME/neoscan-phase1",
    filename="best_phase1.pth"
)

# Note: if clinical CSV not on HF, it will be regenerated in Cell 5
print("✅ Loaded pretrained artifacts from HuggingFace")
print(f"Phase1 weights at: {phase1_path}")

In [ ]:
# Define all model classes: ImageEncoder, ClinicalEncoder, NeonatalTriageModel
import random
import numpy as np
import torch
import torch.nn as nn
from torchvision.models import mobilenet_v3_small

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


class ImageEncoder(nn.Module):
    def __init__(self, phase1_weights_path=None):
        super().__init__()
        base = mobilenet_v3_small(weights=None)
        self.features = base.features
        self.avgpool  = base.avgpool
        if phase1_weights_path:
            # Load Phase1 checkpoint and extract feature/avgpool weights only
            state = torch.load(phase1_weights_path, map_location='cpu')
            if 'model_state_dict' in state:
                state = state['model_state_dict']
            feat_state = {
                k.replace('model.', ''): v
                for k, v in state.items()
                if k.startswith('model.features') or k.startswith('model.avgpool')
            }
            self.load_state_dict(feat_state, strict=False)
            print("✅ ImageEncoder: loaded pretrained feature weights from Phase 1")

    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        return x.flatten(1)  # (B, 576)


class ClinicalEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(8, 64), nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32)
        )
    def forward(self, x):
        return self.mlp(x)  # (B, 32)


class NeonatalTriageModel(nn.Module):
    def __init__(self, phase1_weights_path=None):
        super().__init__()
        self.image_encoder    = ImageEncoder(phase1_weights_path)
        self.clinical_encoder = ClinicalEncoder()
        self.fusion_head = nn.Sequential(
            nn.Linear(608, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 3)
        )
    def forward(self, img, clin):
        img_feat  = self.image_encoder(img)        # (B, 576)
        clin_feat = self.clinical_encoder(clin)    # (B, 32)
        fused = torch.cat([img_feat, clin_feat], dim=1)  # (B, 608)
        return self.fusion_head(fused)             # (B, 3)


# Instantiate with pretrained Phase 1 encoder
model = NeonatalTriageModel(phase1_weights_path=phase1_path).to(device)
total_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {total_params:,}")

In [ ]:
# Multimodal Dataset
# Pairs images from data/frames/ with synthetic clinical records by class
# Falls back to class-tinted dummy 224x224 images if data/frames/ not available
import os, random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from sklearn.model_selection import train_test_split
from collections import Counter
from PIL import Image

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

CLINICAL_RANGES = {
    "ga_weeks":       (24, 42),
    "birth_weight_g": (500, 4000),
    "spo2_pct":       (70, 100),
    "fio2":           (0.21, 1.0),
    "rr_bpm":         (20, 120),
    "hr_bpm":         (80, 220),
    "cpap_cmh2o":     (0, 12),
    "temperature_c":  (34, 39)
}
FEATURE_COLS = list(CLINICAL_RANGES.keys())

def normalize_clinical(row):
    """Min-max normalize using published CLINICAL_RANGES."""
    return [(row[f] - CLINICAL_RANGES[f][0]) / (CLINICAL_RANGES[f][1] - CLINICAL_RANGES[f][0])
            for f in FEATURE_COLS]

# Generate 500 synthetic clinical records if CSV not found from Notebook 1
if not os.path.exists("data/synthetic_clinical.csv"):
    print("Generating synthetic clinical data (Notebook 1 data not found)...")
    rng = np.random.default_rng(42)
    CLASS_PARAMS = {
        0: {"ga":(37,1.5),"bw":(2800,400),"spo2":(97,1.5),"fio2":(0.21,0.02),"rr":(45,8),"hr":(140,15),"cpap":(0,0.5),"temp":(36.8,0.3)},
        1: {"ga":(32,2),"bw":(1800,350),"spo2":(91,3),"fio2":(0.4,0.1),"rr":(65,10),"hr":(155,15),"cpap":(5,1),"temp":(36.5,0.4)},
        2: {"ga":(28,2),"bw":(1100,300),"spo2":(84,5),"fio2":(0.65,0.15),"rr":(80,12),"hr":(165,15),"cpap":(7,1.5),"temp":(36.2,0.5)}
    }
    COUNTS = {0: 200, 1: 150, 2: 150}
    rows = []
    for label, n in COUNTS.items():
        p = CLASS_PARAMS[label]
        for _ in range(n):
            rows.append({
                "ga_weeks":       np.clip(rng.normal(p["ga"][0],   p["ga"][1]),   24, 42),
                "birth_weight_g": np.clip(rng.normal(p["bw"][0],   p["bw"][1]),   500, 4000),
                "spo2_pct":       np.clip(rng.normal(p["spo2"][0], p["spo2"][1]), 70, 100),
                "fio2":           np.clip(rng.normal(p["fio2"][0], p["fio2"][1]), 0.21, 1.0),
                "rr_bpm":         np.clip(rng.normal(p["rr"][0],   p["rr"][1]),   20, 120),
                "hr_bpm":         np.clip(rng.normal(p["hr"][0],   p["hr"][1]),   80, 220),
                "cpap_cmh2o":     np.clip(rng.normal(p["cpap"][0], p["cpap"][1]), 0, 12),
                "temperature_c":  np.clip(rng.normal(p["temp"][0], p["temp"][1]), 34, 39),
                "label": label
            })
    os.makedirs("data", exist_ok=True)
    pd.DataFrame(rows).to_csv("data/synthetic_clinical.csv", index=False)
    print("Regenerated data/synthetic_clinical.csv")

df = pd.read_csv("data/synthetic_clinical.csv")

# Check whether real LUS frames are available
HAS_FRAMES = (os.path.exists("data/frames/Normal") and
              len(os.listdir("data/frames/Normal")) > 0)
CLASS_IMG_DIRS = {0: "data/frames/Normal", 1: "data/frames/Abnormal", 2: "data/frames/Abnormal"}

def get_image_for_class(label, rng_obj):
    """Return path to a real frame for this class, or None if unavailable."""
    d = CLASS_IMG_DIRS[label]
    if HAS_FRAMES and os.path.exists(d):
        files = os.listdir(d)
        if files:
            return os.path.join(d, rng_obj.choice(files))
    return None

sample_rng = np.random.default_rng(42)
records = []
for _, row in df.iterrows():
    records.append({
        "img_path": get_image_for_class(int(row["label"]), sample_rng),
        **{c: row[c] for c in FEATURE_COLS},
        "label": int(row["label"])
    })

# Image transforms
transform_train = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(0.2, 0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
transform_eval = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])


class MultimodalDataset(Dataset):
    def __init__(self, records, transform=None):
        self.records   = records
        self.transform = transform

    def __len__(self): return len(self.records)

    def __getitem__(self, idx):
        item = self.records[idx]
        # Use real frame or fall back to class-tinted noise image
        if item['img_path'] and os.path.exists(item['img_path']):
            img = Image.open(item['img_path']).convert('RGB').resize((224, 224))
        else:
            lo  = 50 + item['label'] * 30
            hi  = 120 + item['label'] * 30
            arr = np.random.randint(lo, hi, (224, 224, 3), dtype=np.uint8)
            img = Image.fromarray(arr)
        if self.transform:
            img = self.transform(img)
        clin = torch.FloatTensor(normalize_clinical(item))
        return img, clin, item['label']


# Stratified 70/15/15 split on original records
labels_arr = [r["label"] for r in records]
idx_all = list(range(len(records)))
idx_train, idx_temp, _, y_temp = train_test_split(
    idx_all, labels_arr, test_size=0.30, stratify=labels_arr, random_state=42)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.5, stratify=y_temp, random_state=42)

train_ds = MultimodalDataset([records[i] for i in idx_train], transform_train)
val_ds   = MultimodalDataset([records[i] for i in idx_val],   transform_eval)
test_ds  = MultimodalDataset([records[i] for i in idx_test],  transform_eval)

# WeightedRandomSampler so minority classes get fair representation
train_labels = [records[i]["label"] for i in idx_train]
counts  = Counter(train_labels)
weights = [1.0 / counts[records[i]["label"]] for i in idx_train]
sampler = WeightedRandomSampler(weights, len(weights))

train_loader = DataLoader(train_ds, batch_size=32, sampler=sampler,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=32, shuffle=False,    num_workers=2)
test_loader  = DataLoader(test_ds,  batch_size=32, shuffle=False,    num_workers=2)
print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
print(f"HAS_FRAMES={HAS_FRAMES} — {'real frames' if HAS_FRAMES else 'dummy images'} used")

In [ ]:
# Training loop
# Freeze image_encoder for first 5 epochs, then unfreeze all
import os, random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, roc_auc_score

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
os.makedirs("models/phase2", exist_ok=True)

EPOCHS        = 30
PATIENCE      = 7
FREEZE_EPOCHS = 5

criterion = nn.CrossEntropyLoss(
    weight=torch.tensor([0.5, 1.0, 1.5]).to(device)
)

# Freeze image encoder; train only fusion head + clinical encoder initially
for p in model.image_encoder.parameters():
    p.requires_grad = False

optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=5e-4, weight_decay=1e-4
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

best_macro_f1  = -1.0
patience_counter = 0

print(f"{'Epoch':>5} {'TrainLoss':>10} {'ValLoss':>10} {'MacroF1':>9} {'ROCAUC':>8}")
print("-" * 48)

for epoch in range(1, EPOCHS + 1):
    # Unfreeze image_encoder after FREEZE_EPOCHS
    if epoch == FREEZE_EPOCHS + 1:
        print(f"  [Epoch {epoch}] Unfreezing image_encoder — training all params")
        for p in model.image_encoder.parameters():
            p.requires_grad = True
        optimizer = torch.optim.Adam(
            model.parameters(), lr=5e-4, weight_decay=1e-4
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS - FREEZE_EPOCHS
        )

    model.train()
    train_loss = 0.0
    for imgs, clins, labels in train_loader:
        imgs, clins, labels = imgs.to(device), clins.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(imgs, clins), labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item() * len(imgs)
    train_loss /= len(train_ds)

    model.eval()
    val_loss = 0.0
    all_preds, all_labels, all_probs = [], [], []
    with torch.no_grad():
        for imgs, clins, labels in val_loader:
            imgs, clins, labels = imgs.to(device), clins.to(device), labels.to(device)
            logits = model(imgs, clins)
            val_loss += criterion(logits, labels).item() * len(imgs)
            probs = F.softmax(logits, dim=1).cpu().numpy()
            all_probs.extend(probs.tolist())
            all_preds.extend(logits.argmax(1).cpu().tolist())
            all_labels.extend(labels.cpu().tolist())
    val_loss  /= len(val_ds)
    macro_f1   = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    try:
        roc_auc = roc_auc_score(all_labels, all_probs, multi_class='ovr', average='macro')
    except Exception:
        roc_auc = float('nan')
    scheduler.step()
    print(f"{epoch:>5} {train_loss:>10.4f} {val_loss:>10.4f} {macro_f1:>9.4f} {roc_auc:>8.4f}")

    if macro_f1 > best_macro_f1:
        best_macro_f1    = macro_f1
        patience_counter = 0
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "best_macro_f1": best_macro_f1},
                   "models/phase2/best_phase2.pth")
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch}")
            break

print(f"\nBest val Macro-F1: {best_macro_f1:.4f}")

In [ ]:
# Test set evaluation
import torch
import torch.nn.functional as F
import numpy as np
from sklearn.metrics import f1_score, classification_report, confusion_matrix, roc_auc_score

# Load best checkpoint
ckpt = torch.load("models/phase2/best_phase2.pth", map_location=device)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()

test_preds, test_labels, test_probs = [], [], []
with torch.no_grad():
    for imgs, clins, labels in test_loader:
        logits = model(imgs.to(device), clins.to(device))
        probs  = F.softmax(logits, dim=1).cpu().numpy()
        test_preds.extend(logits.argmax(1).cpu().tolist())
        test_labels.extend(labels.tolist())
        test_probs.extend(probs.tolist())

target_names = ["Normal", "Moderate RDS", "High RDS"]
print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)
print(classification_report(test_labels, test_preds, target_names=target_names, digits=4))
print(f"Macro-F1:    {f1_score(test_labels, test_preds, average='macro',    zero_division=0):.4f}")
print(f"Weighted-F1: {f1_score(test_labels, test_preds, average='weighted', zero_division=0):.4f}")
try:
    roc_auc = roc_auc_score(test_labels, test_probs, multi_class='ovr', average='macro')
    print(f"ROC-AUC (OvR, macro): {roc_auc:.4f}")
except Exception as e:
    print(f"ROC-AUC error: {e}")

cm = confusion_matrix(test_labels, test_preds)
print("\nConfusion Matrix:")
print(f"{'':15}" + "".join(f"{n:>14}" for n in target_names))
for i, row in enumerate(cm):
    print(f"{target_names[i]:15}" + "".join(f"{v:>14}" for v in row))

print("\nTarget: Macro-F1 \u2265 0.61, ROC-AUC \u2265 0.76")

In [ ]:
# --- HUGGINGFACE CHECKPOINT SAVE ---
# Setup: Colab left panel → key icon → Add secret: Name=HF_TOKEN, Value=your_write_token
import os, json
from huggingface_hub import HfApi

os.makedirs("models/phase2", exist_ok=True)

api     = HfApi()
repo_id = "YOUR_HF_USERNAME/neoscan-phase2"  # <-- CHANGE THIS

api.create_repo(repo_id=repo_id, exist_ok=True)
api.upload_file(
    path_or_fileobj="models/phase2/best_phase2.pth",
    path_in_repo="best_phase2.pth",
    repo_id=repo_id
)

config = {
    "model": "NeonatalTriageModel",
    "architecture": "MobileNetV3Small + ClinicalMLP late fusion",
    "image_dim": 576, "clinical_dim": 32, "fusion_dim": 608,
    "num_classes": 3,
    "classes": ["Normal", "Moderate RDS", "High RDS"],
    "clinical_features": [
        "ga_weeks", "birth_weight_g", "spo2_pct", "fio2",
        "rr_bpm", "hr_bpm", "cpap_cmh2o", "temperature_c"
    ],
    "clinical_ranges": {
        "ga_weeks": [24, 42], "birth_weight_g": [500, 4000],
        "spo2_pct": [70, 100], "fio2": [0.21, 1.0],
        "rr_bpm": [20, 120], "hr_bpm": [80, 220],
        "cpap_cmh2o": [0, 12], "temperature_c": [34, 39]
    }
}
with open("models/phase2/phase2_config.json", "w") as f:
    json.dump(config, f, indent=2)
api.upload_file(
    path_or_fileobj="models/phase2/phase2_config.json",
    path_in_repo="phase2_config.json",
    repo_id=repo_id
)
print(f"✅ Phase 2 saved to: https://huggingface.co/{repo_id}")

## 🔁 RETURNING AFTER TRAINING?

Run ONLY these cells:
- Cell 1 — Install (always needed)
- Cell 2 — GPU check
- Cell 4 — Define all model classes (NeonatalTriageModel etc.)
- Cell 9b (below) — Load Phase 2 + Phase 1 from HuggingFace
- Cell 10 — Launch Gradio demo

SKIP: Cells 3, 5, 6, 7 (data + training already done)

Load Phase 2 snippet:
```python
from huggingface_hub import hf_hub_download
path = hf_hub_download(repo_id="YOUR_HF_USERNAME/neoscan-phase2", filename="best_phase2.pth")
model = NeonatalTriageModel().to(device)
model.load_state_dict(torch.load(path, map_location=device))
model.eval()
print("✅ Phase 2 loaded")
```

Also load Phase 1 for the binary tab in Gradio:
```python
from torchvision.models import mobilenet_v3_small
import torch.nn as nn
class Phase1Model(nn.Module):
    def __init__(self):
        super().__init__()
        base = mobilenet_v3_small(weights=None)
        base.classifier[3] = nn.Linear(base.classifier[3].in_features, 2)
        self.model = base
    def forward(self, x): return self.model(x)

p1_path = hf_hub_download(repo_id="YOUR_HF_USERNAME/neoscan-phase1", filename="best_phase1.pth")
phase1_model = Phase1Model().to(device)
phase1_model.load_state_dict(torch.load(p1_path, map_location=device))
phase1_model.eval()
print("✅ Phase 1 loaded")
```

In [ ]:
# Load Phase 2 from HuggingFace (run when returning after training)
import torch
import torch.nn as nn
from huggingface_hub import hf_hub_download
from torchvision.models import mobilenet_v3_small

# Load Phase 2 fusion model
p2_path = hf_hub_download(
    repo_id="YOUR_HF_USERNAME/neoscan-phase2",
    filename="best_phase2.pth"
)
model = NeonatalTriageModel().to(device)
ckpt2 = torch.load(p2_path, map_location=device)
if 'model_state_dict' in ckpt2:
    model.load_state_dict(ckpt2['model_state_dict'])
else:
    model.load_state_dict(ckpt2)
model.eval()

# Load Phase 1 binary model for the Gradio binary tab
class Phase1Model(nn.Module):
    def __init__(self):
        super().__init__()
        base = mobilenet_v3_small(weights=None)
        base.classifier[3] = nn.Linear(base.classifier[3].in_features, 2)
        self.model = base
    def forward(self, x):
        return self.model(x)

p1_path = hf_hub_download(
    repo_id="YOUR_HF_USERNAME/neoscan-phase1",
    filename="best_phase1.pth"
)
phase1_model = Phase1Model().to(device)
ckpt1 = torch.load(p1_path, map_location=device)
if 'model_state_dict' in ckpt1:
    phase1_model.load_state_dict(ckpt1['model_state_dict'])
else:
    phase1_model.load_state_dict(ckpt1)
phase1_model.eval()
print("✅ Both models loaded from HuggingFace")

In [ ]:
# Full Gradio Demo — 3 tabs: Phase 1 binary, Phase 2 multimodal, About
import gradio as gr
import torch
import torch.nn.functional as F
import numpy as np
from torchvision import transforms
from PIL import Image

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

CLASSES = ["🟢 Normal", "🟡 Moderate RDS", "🔴 High RDS"]
RANGES = {
    "ga_weeks":       (24, 42),
    "birth_weight_g": (500, 4000),
    "spo2_pct":       (70, 100),
    "fio2":           (0.21, 1.0),
    "rr_bpm":         (20, 120),
    "hr_bpm":         (80, 220),
    "cpap_cmh2o":     (0, 12),
    "temperature_c":  (34, 39)
}

def norm(val, key):
    lo, hi = RANGES[key]
    return (val - lo) / (hi - lo)

def predict_phase1(image):
    """Phase 1: binary Normal vs Abnormal classification."""
    if image is None:
        return {"Normal": 0.5, "Abnormal (Possible RDS)": 0.5}
    img = transform(Image.fromarray(image).convert("RGB")).unsqueeze(0).to(device)
    with torch.no_grad():
        prob = F.softmax(phase1_model(img), dim=1)[0].cpu()
    return {"Normal": float(prob[0]), "Abnormal (Possible RDS)": float(prob[1])}

def predict_phase2(image, ga, bw, spo2, fio2, rr, hr, cpap, temp):
    """Phase 2: 3-class RDS severity triage using image + clinical data."""
    if image is None:
        return {c: 1/3 for c in CLASSES}
    img = transform(Image.fromarray(image).convert("RGB")).unsqueeze(0).to(device)
    clin = torch.FloatTensor([[
        norm(ga,   "ga_weeks"),
        norm(bw,   "birth_weight_g"),
        norm(spo2, "spo2_pct"),
        norm(fio2, "fio2"),
        norm(rr,   "rr_bpm"),
        norm(hr,   "hr_bpm"),
        norm(cpap, "cpap_cmh2o"),
        norm(temp, "temperature_c")
    ]]).to(device)
    with torch.no_grad():
        prob = F.softmax(model(img, clin), dim=1)[0].cpu()
    return {c: float(p) for c, p in zip(CLASSES, prob)}

with gr.Blocks(title="NeoScan AI") as demo:
    gr.Markdown(
        "# 🏥 NeoScan AI\n"
        "**Neonatal LUS RDS Triage System — SRHU Dehradun**\n\n"
        "*For research purposes only. Not for clinical use.*"
    )

    with gr.Tab("Phase 1 — Binary Screen"):
        img1 = gr.Image(label="Upload LUS Image")
        btn1 = gr.Button("Analyze", variant="primary")
        out1 = gr.Label(label="Result", num_top_classes=2)
        btn1.click(predict_phase1, inputs=img1, outputs=out1)

    with gr.Tab("Phase 2 — Multimodal Triage"):
        with gr.Row():
            img2 = gr.Image(label="Upload LUS Image")
            with gr.Column():
                ga   = gr.Slider(24,   42,   value=32,   step=0.5,  label="GA (weeks)")
                bw   = gr.Slider(500,  4000, value=1500, step=50,   label="Birth Weight (g)")
                spo2 = gr.Slider(70,   100,  value=90,   step=1,    label="SpO\u2082 (%)")
                fio2 = gr.Slider(0.21, 1.0,  value=0.4,  step=0.01, label="FiO\u2082")
                rr   = gr.Slider(20,   120,  value=65,   step=1,    label="RR (bpm)")
                hr   = gr.Slider(80,   220,  value=155,  step=1,    label="HR (bpm)")
                cpap = gr.Slider(0,    12,   value=5,    step=0.5,  label="CPAP (cmH\u2082O)")
                temp = gr.Slider(34.0, 39.0, value=36.5, step=0.1,  label="Temperature (\u00b0C)")
        btn2 = gr.Button("Run Triage", variant="primary")
        out2 = gr.Label(label="RDS Severity", num_top_classes=3)
        btn2.click(predict_phase2,
                   inputs=[img2, ga, bw, spo2, fio2, rr, hr, cpap, temp],
                   outputs=out2)

    with gr.Tab("About"):
        gr.Markdown("""
## NeoScan AI
**Institution:** SRHU, Dehradun
**Architecture:** MobileNetV3-Small + ClinicalMLP (Late Fusion)
**Classes:** Normal | Moderate RDS | High RDS
**Dataset:** covid19_ultrasound + synthetic neonatal domain adaptation
**Clinical features:** GA, Birth Weight, SpO\u2082, FiO\u2082, RR, HR, CPAP, Temperature
        """)

demo.launch(share=True)
print("✅ Gradio running — copy the gradio.live URL above to share!")

In [ ]:
# Download Phase 2 .pth backup to PC
from google.colab import files
files.download("models/phase2/best_phase2.pth")